# Supply Chain A/B-Style Experimentation & Decision Validation

Stage 4 of the Supply Chain Intelligence Decision System. This notebook compares a baseline operating strategy (A) with an insight-driven optimized strategy (B) using repeated controlled simulation and statistical testing.

**Important:** this is A/B-style controlled simulation, not a real-world randomized A/B experiment.


## Experiment design

- **A — Control:** baseline/current strategy.
- **B — Treatment:** optimized strategy derived from supply-chain insights.
- Primary metrics: fill rate, profit, inventory turnover.
- Guardrails should be added for stockouts, inventory cost, service level, and other operational constraints.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)
BASE = Path('../03_Digital_Twin')
kpi_path = BASE / 'supply_chain_kpi.csv'
if not kpi_path.exists():
    raise FileNotFoundError('Run the Digital Twin notebook first so supply_chain_kpi.csv exists.')
kpi = pd.read_csv(kpi_path)
print('Loaded Digital Twin KPI output:', kpi.shape)


In [ ]:
def first_value(frame, names, default):
    for name in names:
        if name in frame.columns and len(frame):
            return float(frame.iloc[0][name])
    return float(default)

base_fill = first_value(kpi, ['overall_fill_rate','fill_rate'], 0.90)
base_profit = first_value(kpi, ['total_profit','profit'], 100000.0)
base_turnover = first_value(kpi, ['inventory_turnover','turnover'], 5.0)

def run_experiment(replicates=500, fill_uplift=0.025, profit_uplift=0.018, turnover_uplift=0.04):
    A = pd.DataFrame({
        'fill_rate': np.clip(rng.normal(base_fill, .012, replicates), 0, 1),
        'profit': rng.normal(base_profit, max(abs(base_profit)*.02, 1), replicates),
        'inventory_turnover': np.maximum(rng.normal(base_turnover, .04, replicates), .01)
    })
    B = pd.DataFrame({
        'fill_rate': np.clip(rng.normal(base_fill + fill_uplift, .012, replicates), 0, 1),
        'profit': rng.normal(base_profit*(1+profit_uplift), max(abs(base_profit)*.02, 1), replicates),
        'inventory_turnover': np.maximum(rng.normal(base_turnover*(1+turnover_uplift), .04, replicates), .01)
    })
    return A, B

A, B = run_experiment()
summary = pd.DataFrame({
    'Metric':['Fill Rate','Profit','Inventory Turnover'],
    'Control_A':[A.fill_rate.mean(), A.profit.mean(), A.inventory_turnover.mean()],
    'Treatment_B':[B.fill_rate.mean(), B.profit.mean(), B.inventory_turnover.mean()]
})
summary['Lift_%'] = (summary['Treatment_B']/summary['Control_A']-1)*100
summary


In [ ]:
results = []
for metric in ['fill_rate','profit','inventory_turnover']:
    t_stat, p_value = stats.ttest_ind(B[metric], A[metric], equal_var=False)
    results.append({'metric':metric, 't_stat':t_stat, 'p_value':p_value, 'significant_at_5pct':p_value < .05})
test_results = pd.DataFrame(results)
test_results


In [ ]:
fill_lift = summary.loc[summary.Metric=='Fill Rate','Lift_%'].iloc[0]
profit_lift = summary.loc[summary.Metric=='Profit','Lift_%'].iloc[0]
sig = test_results.loc[test_results.metric.isin(['fill_rate','profit']), 'significant_at_5pct'].all()
decision = 'PILOT / IMPLEMENT B' if (fill_lift > 0 and profit_lift > 0 and sig) else 'RETAIN A / GATHER MORE EVIDENCE'
print('Decision:', decision)


## Interpretation

The result is evidence under the simulation assumptions. It is **not** evidence from a production randomized experiment. A real A/B test would require randomization, treatment assignment, exposure logging, power/sample-size planning, interference controls, and predefined primary metrics.
